In [2]:
import os
import requests
import pandas as pd
import io
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table
from dask import delayed, compute
from dask.distributed import Client
import warnings
warnings.filterwarnings('ignore')

In [3]:
class SistemaPlanetario:
    def __init__(self, m1, m2, d):
        self.m1 = m1 * 1.989E30  # masa estrella en kg
        self.m2 = m2 * 5.972E24  # masa planeta en kg (Ingresa en Masas Terrestres)
        self.mu2 = self.m2 / (self.m1 + self.m2)
        self.mu1 = 1.0 - self.mu2
        self.distanciaPrimarios = d * 1.496E8  # AU -> km

        # Parámetros de simulación
        self.m = 100
        self.n = self.m
        self.a = 0.2
        self.b = self.a
        self.tiempo = 500
        self.M = 1000

        # Listas de resultados
        self.estable_x = []
        self.estable_y = []
        self.inestable_x = []
        self.inestable_y = []
        self.colision_m1x = []
        self.colision_m1y = []
        self.colision_m2x = []
        self.colision_m2y = []

    # Función del sistema
    def f1(self, t, v):
        x, y, xp, yp = v
        r1p = ((x + self.mu2)**2 + y**2)**(-3/2.)
        r2p = ((x - self.mu1)**2 + y**2)**(-3/2.)
        dv1 = xp + y
        dv2 = yp - x
        dv3 = yp - self.mu1 * (x + self.mu2) * r1p - self.mu2 * (x - self.mu1) * r2p
        dv4 = -xp - self.mu1 * y * r1p - self.mu2 * y * r2p
        return np.array([dv1, dv2, dv3, dv4])

    # Hamiltoniano
    def ham(self, s):
        x, y, px, py = s
        r1p = ((x + self.mu2)**2 + y**2)**0.5
        r2p = ((x - self.mu1)**2 + y**2)**0.5
        return (0.5 * (px**2 + py**2) + y * px - x * py 
                - self.mu1 / r1p - self.mu2 / r2p)

    # Runge-Kutta de 6to orden
    def RungeKutaS6(self, t0, tn, x0, n):
        t = np.linspace(t0, tn, n + 1)
        m = len(x0)
        x = np.zeros((m, n + 1))
        x[:, 0] = x0
        h = (tn - t0) / n
        TOL = 0.01

        for i in range(1, n + 1):
            k1 = self.f1(t[i-1], x[:, i-1])
            k2 = self.f1(t[i-1] + h*(1/3.), x[:, i-1] + (1/3.)*k1*h)
            k3 = self.f1(t[i-1] + h*(2/3.), x[:, i-1] + (2/3)*k2*h)
            k4 = self.f1(t[i-1] + h*(1/3), x[:, i-1] + (1/12)*k1*h + (1/3)*k2*h - (1/12)*h*k3)
            k5 = self.f1(t[i-1] + h*(5/6.), x[:, i-1] + (25/48.)*k1*h - (55/24.)*k2*h + (35/48.)*k3*h + k4*(15/8.)*h)
            k6 = self.f1(t[i-1] + h*(1/6.), x[:, i-1] + (3/20)*k1*h - (11/24)*k2*h - (1/8)*k3*h + (1/2)*k4*h + (1/10)*h*k5)
            k7 = self.f1(t[i], x[:, i-1] - (261/260)*k1*h + (33/13)*k2*h + (43/156)*k3*h - (118/39)*k4*h + (32/195)*k5*h + (80/39)*k6*h)
            x[:, i] = x[:, i-1] + ((13/200)*k1 + (11/40.)*k3 + (11/40.)*k4 + (4/25)*k5 + (4/25)*k6 + (13/200)*k7)*h

            if x[1, i] < 0:
                j = 1
                break
            elif (np.abs(x[0, i] - self.mu2) < TOL and np.abs(x[1, i]) < TOL):
                j = 2
                break
            elif (np.abs(x[0, i] - self.mu1) < TOL and np.abs(x[1, i]) < TOL):
                j = 3
                break
        else:
            j = 4

        return j, x

    # Crear malla de puntos
    def malla(self, m, n, a, b):
        x = np.linspace(-a, a, n)
        y = np.linspace(-b, b, m)
        xx, yy = np.meshgrid(x, y)
        points = np.c_[xx.ravel(), yy.ravel()]
        return points

    # Simulación con Dask
    def simular(self, client=None, chunk_size=100):
        points = self.malla(self.m, self.n, self.a, self.b)
        total_points = len(points)

        @delayed
        def simular_lote(lote):
            resultados_lote = []
            for point in lote:
                x0 = 0.5 - self.mu2 + point[0]
                y0 = (3**0.5) * 0.5 + point[1]
                f0 = [x0, y0, -y0, x0]
                j, x = self.RungeKutaS6(0, self.tiempo, f0, self.M)
                resultados_lote.append((j, x0, y0))
            return resultados_lote

        lotes = [points[i:i+chunk_size] for i in range(0, total_points, chunk_size)]
        tareas = [simular_lote(lote) for lote in lotes]

        if client is not None:
            resultados = client.compute(tareas, sync=True)
        else:
            resultados = compute(*tareas)

        for lote in resultados:
            for j, x0, y0 in lote:
                if j == 1:
                    self.inestable_x.append(x0)
                    self.inestable_y.append(y0)
                elif j == 2:
                    self.colision_m1x.append(x0)
                    self.colision_m1y.append(y0)
                elif j == 3:
                    self.colision_m2x.append(x0)
                    self.colision_m2y.append(y0)
                else:
                    self.estable_x.append(x0)
                    self.estable_y.append(y0)

    # Guardar resultados y graficar (Correctamente indentado)
    def guardar_resultados(self, directorio, nombre="zonas", graficar=True):
        os.makedirs(directorio, exist_ok=True)

        Table([self.estable_x, self.estable_y], names=('x', 'y')).write(
            os.path.join(directorio, f"{nombre}_estables.txt"),
            format='ascii.commented_header', overwrite=True)
        Table([self.inestable_x, self.inestable_y], names=('x', 'y')).write(
            os.path.join(directorio, f"{nombre}_inestables.txt"),
            format='ascii.commented_header', overwrite=True)
        Table([self.colision_m1x, self.colision_m1y], names=('x', 'y')).write(
            os.path.join(directorio, f"{nombre}_colision_m1.txt"),
            format='ascii.commented_header', overwrite=True)
        Table([self.colision_m2x, self.colision_m2y], names=('x', 'y')).write(
            os.path.join(directorio, f"{nombre}_colision_m2.txt"),
            format='ascii.commented_header', overwrite=True)

        estable_x = len(self.estable_x)
        inestable_x = len(self.inestable_x)
        colision_m1 = len(self.colision_m1x)
        colision_m2 = len(self.colision_m2x)

        deltaA = (2 * self.a) / self.n
        deltaB = (2 * self.b) / self.m
        deltaKmA = deltaA * self.distanciaPrimarios
        deltaKmB = deltaB * self.distanciaPrimarios
        area = deltaKmA * deltaKmB

        area_est = area * estable_x
        area_inest = area * inestable_x
        area_colM1 = area * colision_m1
        area_colM2 = area * colision_m2

        resumen_path = os.path.join(directorio, f"{nombre}_resumen.txt")
        with open(resumen_path, "w", encoding='utf-8') as f:
            f.write("=== Resumen de resultados ===\n")
            f.write(f"Total estables: {estable_x}\n")
            f.write(f"Total inestables: {inestable_x}\n")
            f.write(f"Total colisión m1: {colision_m1}\n")
            f.write(f"Total colisión m2: {colision_m2}\n\n")
            f.write("=== Áreas en km² ===\n")
            f.write("Área total de la grilla: {:.2e} km²\n".format(area))
            f.write("Área estable: {:.2e} km²\n".format(area_est))
            f.write("Área inestable: {:.2e} km²\n".format(area_inest))
            f.write("Área de colisión con m1: {:.2e} km²\n".format(area_colM1))
            f.write("Área de colisión con m2: {:.2e} km²\n".format(area_colM2))

        if graficar:
            plt.scatter(self.estable_x, self.estable_y, color='blue', s=10, label='Estables')
            plt.scatter(self.inestable_x, self.inestable_y, color='silver', s=10, label='Inestables')
            plt.scatter(self.colision_m1x, self.colision_m1y, color='green', s=10, label='Colisión m1')
            plt.scatter(self.colision_m2x, self.colision_m2y, color='brown', s=10, label='Colisión m2')
            plt.scatter(0.5 - self.mu2, np.sqrt(3)/2, marker='v', color='red', label='L4')
            plt.xlabel('x [adim]')
            plt.ylabel('y [adim]')
            plt.legend()
            plt.title(f"Mapa de estabilidad - {nombre}")
            ruta_pdf = os.path.join(directorio, f"{nombre}_mapa_estabilidad.pdf")
            plt.savefig(ruta_pdf, format="pdf", bbox_inches="tight")
            plt.close()

In [4]:
def ejecutar_pipeline_completo(tipo_espectral, masa_min, masa_max, client):
    """
    Extrae datos de la NASA, los limpia, crea la estructura de carpetas
    e inyecta los datos directamente en la simulación astrofísica.
    """
    url_base = "https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
    tipo_esp_sql = tipo_espectral.replace("'", "''")
    
    query = f"""
    SELECT hostname AS estrella, sy_pnum, sy_dist, st_spectype, st_teff, st_rad, st_mass, 
           st_teff_reflink, pl_name, pl_orbper, pl_orbsmax, pl_rade, pl_bmasse, pl_bmassj, pl_msinij
    FROM pscomppars
    WHERE sy_pnum BETWEEN 2 AND 8
      AND st_spectype LIKE '%{tipo_esp_sql}%'
      AND (pl_bmassj BETWEEN {masa_min} AND {masa_max} OR pl_msinij BETWEEN {masa_min} AND {masa_max})
    """
    
    print(f"📡 Consultando base de datos NASA... (Espectro: {tipo_espectral})")
    respuesta = requests.get(url_base, params={"query": query, "format": "csv"})
    if respuesta.status_code != 200: 
        print("Error al contactar la API.")
        return
    
    df_completo = pd.read_csv(io.StringIO(respuesta.text))
    if df_completo.empty:
        print("No se encontraron sistemas con esos criterios.")
        return
    
    # LIMPIEZA: Quedarse con el paper más reciente por estrella
    df_completo['Año_Ref'] = df_completo['st_teff_reflink'].str.extract(r'(\d{4})').astype(float)
    df_star = df_completo.sort_values(by=['estrella', 'Año_Ref'], ascending=[True, False]).drop_duplicates(subset=['estrella'], keep='first')

    # CREACIÓN DEL DIRECTORIO BASE
    base_dir = f"Resultados_Estrellas_{tipo_espectral}"
    os.makedirs(base_dir, exist_ok=True)
    print(f"📁 Directorio raíz creado: {base_dir}")
    
    # ITERAR SOBRE CADA ESTRELLA ENCONTRADA
    for _, star in df_star.iterrows():
        est_nombre = star['estrella'].replace(" ", "_")
        m_estrella = star['st_mass']
        
        if pd.isna(m_estrella): continue # Saltamos si no hay masa estelar conocida
            
        print(f"\n⚙️ Procesando sistema: {est_nombre} (Masa: {m_estrella} M_sol)")
        
        # Filtrar planetas correspondientes a esta estrella
        planets = df_completo[df_completo['estrella'] == star['estrella']]
        
        for _, pl in planets.iterrows():
            distancia = pl['pl_orbsmax']
            
            # FILTRO: Necesitamos la distancia para simular
            if pd.isna(distancia): 
                print(f"  [!] Saltando {pl['pl_name']} (Falta parámetro de distancia orbital)")
                continue
            
            # CONVERSIÓN A MASAS TERRESTRES
            # Priorizamos la masa en tierras, si no, convertimos de Júpiter a Tierras (* 317.828)
            if pd.notna(pl['pl_bmasse']):
                masa_tierra = pl['pl_bmasse']
            elif pd.notna(pl['pl_bmassj']):
                masa_tierra = pl['pl_bmassj'] * 317.828
            elif pd.notna(pl['pl_msinij']):
                masa_tierra = pl['pl_msinij'] * 317.828
            else:
                print(f"  [!] Saltando {pl['pl_name']} (Falta parámetro de masa)")
                continue
                
            # CREACIÓN DE DIRECTORIO PARA EL PLANETA
            nombre_planeta_limpio = pl['pl_name'].replace(" ", "_")
            path = os.path.join(base_dir, est_nombre, nombre_planeta_limpio)
            os.makedirs(path, exist_ok=True)
            
            # EXPORTAR README DEL PLANETA
            readme_path = os.path.join(path, f"README_{nombre_planeta_limpio}.txt")
            with open(readme_path, "w", encoding="utf-8") as f:
                f.write(f"SISTEMA: {star['estrella']}\n")
                f.write(f"PLANETA: {pl['pl_name']}\n")
                f.write("=" * 40 + "\n")
                f.write(f"Masa de la Estrella (Masas Solares): {m_estrella}\n")
                f.write(f"Temperatura Estrella (K): {star['st_teff']}\n")
                f.write(f"Masa del Planeta (Masas Terrestres): {masa_tierra:.4f}\n")
                f.write(f"Semi-eje mayor de la órbita (UA): {distancia}\n")
                f.write(f"Periodo Orbital (días): {pl['pl_orbper']}\n")
                f.write(f"Referencia: {star['st_teff_reflink']}\n")

            # --- SIMULACIÓN FÍSICA ---
            print(f"  -> Simulando planeta {pl['pl_name']} ...")
            sistema = SistemaPlanetario(m_estrella, masa_tierra, distancia)
            sistema.simular(client=client)
            sistema.guardar_resultados(path, nombre=f"{est_nombre}_{nombre_planeta_limpio}")
            print(f"  -> ✔️ Resultados guardados en: {path}")

In [ ]:
from PIL import GimpGradientFile
from PIL import GimpGradientFile
if __name__ == "__main__":
    # 1. Configuración de Filtros de Investigación
    ESPECTRO = "G"
    M_MIN_JUP = 0.0001
    M_MAX_JUP = 10.0
    
    # 2. Iniciar cliente de procesamiento paralelo Dask
    print("Iniciando workers de Dask...")
    cliente_dask = Client(n_workers=4)
    
    # 3. Lanzar pipeline
    ejecutar_pipeline_completo(ESPECTRO, M_MIN_JUP, M_MAX_JUP, client=cliente_dask)

Iniciando workers de Dask...
📡 Consultando base de datos NASA... (Espectro: F)
📁 Directorio raíz creado: Resultados_Estrellas_F

⚙️ Procesando sistema: DMPP-1 (Masa: 1.21 M_sol)
  -> Simulando planeta DMPP-1 e ...
  -> ✔️ Resultados guardados en: Resultados_Estrellas_F\DMPP-1\DMPP-1_e
  -> Simulando planeta DMPP-1 b ...
  -> ✔️ Resultados guardados en: Resultados_Estrellas_F\DMPP-1\DMPP-1_b
  -> Simulando planeta DMPP-1 c ...
  -> ✔️ Resultados guardados en: Resultados_Estrellas_F\DMPP-1\DMPP-1_c
  -> Simulando planeta DMPP-1 d ...
  -> ✔️ Resultados guardados en: Resultados_Estrellas_F\DMPP-1\DMPP-1_d

⚙️ Procesando sistema: DMPP-2 (Masa: 1.41 M_sol)
  -> Simulando planeta DMPP-2 b ...
  -> ✔️ Resultados guardados en: Resultados_Estrellas_F\DMPP-2\DMPP-2_b
  -> Simulando planeta DMPP-2 c ...
  -> ✔️ Resultados guardados en: Resultados_Estrellas_F\DMPP-2\DMPP-2_c
  -> Simulando planeta DMPP-2 d ...
  -> ✔️ Resultados guardados en: Resultados_Estrellas_F\DMPP-2\DMPP-2_d

⚙️ Procesando s